# Reducir a 30 variables o dimensiones meteorológicas y epidemiológicas

Aquí tienes el script de Python para cargar el archivo desde la ubicación especificada, generar los rezagos (lags) desde la semana 1 hasta la semana 12 para la variable `casos_dengue` de forma automatizada, y guardar el nuevo conjunto de datos.



He utilizado un bucle para crear dinámicamente las 12 nuevas columnas (`casos_dengue_lag_1`, `casos_dengue_lag_2`, ..., `casos_dengue_lag_12`). Al final, el script te guardará el resultado en un nuevo archivo Excel en la misma carpeta para que lo tengas listo para tus entrenamientos.




In [1]:
import pandas as pd
import numpy as np

# 1. Definir rutas de archivos
ruta_entrada = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\1_raw\1_meteo_epi_rezagos.xlsx"
ruta_salida = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx"

print("Cargando el archivo de datos original...")
df = pd.read_excel(ruta_entrada)

# 2. Asegurar el preprocesamiento temporal e índice
df['fecha'] = pd.to_datetime(df['fecha'])
df.set_index('fecha', inplace=True)
df = df.sort_index()

# 3. Generar de forma automatizada los rezagos (lags) de 1 a 12 semanas
print("Generando rezagos de 1 a 12 semanas para 'casos_dengue'...")
for i in range(1, 13):
    nombre_columna = f'casos_dengue_lag_{i}'
    df[nombre_columna] = df['casos_dengue'].shift(i)

# 4. Manejo de valores nulos (Opcional pero Recomendado)
# Nota: Al desplazar la serie hasta 12 semanas, las primeras 12 filas del dataset 
# no tendrán historia previa y se rellenarán con NaN.
# Si deseas eliminar esas 12 filas iniciales para limpiar el dataset, desmarca la siguiente línea:
df.dropna(inplace=True)

# 5. Guardar el nuevo dataframe con las dimensiones expandidas
print(f"Guardando el nuevo archivo en: {ruta_salida}")
# Reseteamos el índice para que la columna 'fecha' vuelva a guardarse como columna normal en el Excel
df.reset_index().to_excel(ruta_salida, index=False)

print("\n=== PROCESO COMPLETADO EXCOLSAMENTE ===")
print(f"Dimensiones finales del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.")


Cargando el archivo de datos original...
Generando rezagos de 1 a 12 semanas para 'casos_dengue'...
Guardando el nuevo archivo en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx


C:\Users\marco\AppData\Local\Temp\ipykernel_25300\2858720489.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[nombre_columna] = df['casos_dengue'].shift(i)
C:\Users\marco\AppData\Local\Temp\ipykernel_25300\2858720489.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.reset_index().to_excel(ruta_salida, index=False)



=== PROCESO COMPLETADO EXCOLSAMENTE ===
Dimensiones finales del dataset: 237 filas y 126 columnas.



# Notas sobre lo que hace este script:

1. **`df['casos_dengue'].shift(i)`**: Esta función desplaza los valores de la columna hacia abajo cronológicamente en $i$ posiciones. Por ejemplo, en la fila de la semana actual, `casos_dengue_lag_1` contendrá el valor real que ocurrió la semana pasada.
2. **Efecto colateral de los Lags**: Las primeras 12 semanas de tu dataset (al inicio de 2021) se llenarán con valores nulos (`NaN`) en los rezagos más largos porque matemáticamente no hay registros previos a esa fecha. Si los algoritmos te dan error por valores nulos, descomenta la línea `df.dropna(inplace=True)` del script para purgar automáticamente esas semanas iniciales.